In [0]:
%pip install -U langchain langchain-community databricks-langchain mlflow
 
dbutils.library.restartPython()

In [0]:
import mlflow

# Set experiment for better organization
mlflow.set_experiment("/Users/naval.datamaster@gmail.com/model")

# Enable autologging BEFORE creating the LangChain client
mlflow.langchain.autolog()

In [0]:
from databricks_langchain import ChatDatabricks
from langchain_core.prompts import PromptTemplate

In [0]:
from databricks_langchain import ChatDatabricks
from langchain_core.prompts import PromptTemplate

# Initialize the LLM
llm = ChatDatabricks(model="databricks-gemma-3-12b")

# Create the prompt template
prompt_template = PromptTemplate(
    input_variables=["city", "month", "budget", "days", "num_people"],
    template="""You are an expert travel planner and guide.
    
    Create a detailed travel itinerary for the following trip:
    - Destination: {city}
    - Travel Month: {month}
    - Budget: ${budget} USD
    - Duration: {days} days
    - Number of Travelers: {num_people} people
    
    Please provide:
    1. Day-by-day itinerary with must-visit attractions
    2. Recommended accommodations within the budget
    3. Local cuisine and restaurant suggestions
    4. Transportation tips
    5. Budget breakdown (accommodation, food, activities, transport)
    6. Important travel tips and cultural considerations
    
    Make the itinerary practical, exciting, and suitable for the group size and budget.
    """
)

print("=" * 60)
print("🌍 TRAVEL GUIDE ITINERARY APP 🌍")
print("=" * 60)
print()

# Collect user inputs
city = input("Enter destination city: ")
month = input("Enter travel month: ")
budget = input("Enter total budget (USD): ")
days = input("Enter number of days: ")
num_people = input("Enter number of people: ")

print("\n" + "=" * 60)
print("🔄 Generating your personalized travel itinerary...")
print("=" * 60 + "\n")

# Generate the itinerary
if city and month and budget and days and num_people:
    formatted_prompt = prompt_template.format(
        city=city,
        month=month,
        budget=budget,
        days=days,
        num_people=num_people
    )
    
    response = llm.invoke(formatted_prompt)
    print(response.content)
    
    print("\n" + "=" * 60)
    print("✅ Itinerary generated successfully!")
    print("=" * 60)
else:
    print("⚠️ Please provide all required information.")

In [0]:
%%writefile /tmp/travel_itinerary_model.py

import mlflow
from databricks_langchain import ChatDatabricks
from langchain_core.prompts import PromptTemplate


class TravelPlannerModel(mlflow.pyfunc.PythonModel):

    def load_context(self, context):

        self.llm = ChatDatabricks(
            model="databricks-gemma-3-12b"
        )

        self.prompt_template = PromptTemplate(
            input_variables=[
                "city",
                "month",
                "budget",
                "days",
                "num_people"
            ],
            template="""You are an expert travel planner and guide.

Create a detailed travel itinerary for the following trip:

Destination: {city}
Travel Month: {month}
Budget: ${budget} USD
Duration: {days} days
Number of Travelers: {num_people} people

Please provide:

1. Day-by-day itinerary with must-visit attractions
2. Recommended accommodations within the budget
3. Local cuisine and restaurant suggestions
4. Transportation tips
5. Budget breakdown
6. Important travel tips and cultural considerations

Make the itinerary practical, exciting, and suitable for the group size and budget.
"""
        )

    def predict(self, context, model_input):

        results = []

        for _, row in model_input.iterrows():

            prompt = self.prompt_template.format(
                city=row["city"],
                month=row["month"],
                budget=row["budget"],
                days=row["days"],
                num_people=row["num_people"]
            )

            response = self.llm.invoke(prompt)

            results.append(response.content)

        return results


# This is the important line
mlflow.models.set_model(TravelPlannerModel())

In [0]:
mlflow.models.set_model(TravelPlannerModel())

In [0]:
import mlflow
from mlflow.models import ModelSignature
from mlflow.types import Schema, ColSpec

mlflow.set_registry_uri("databricks-uc")

# -----------------------------
# Define model input schema
# -----------------------------

input_schema = Schema([
    ColSpec("string", "city"),
    ColSpec("string", "month"),
    ColSpec("double", "budget"),
    ColSpec("long", "days"),
    ColSpec("long", "num_people")
])

# -----------------------------
# Define model output schema
# -----------------------------

output_schema = Schema([
    ColSpec("string")
])

signature = ModelSignature(
    inputs=input_schema,
    outputs=output_schema
)

# -----------------------------
# Unity Catalog model name
# -----------------------------

uc_model_name = "dev.bronze.travel_itinerary"

# -----------------------------
# Log and register
# -----------------------------

with mlflow.start_run() as run:

    model_info = mlflow.pyfunc.log_model(
        artifact_path="travel_itinerary",

        python_model="/tmp/travel_itinerary_model.py",

        signature=signature,

        registered_model_name=uc_model_name
    )

    print("Run ID:", run.info.run_id)
    print("Model URI:", model_info.model_uri)

In [0]:
import mlflow
import pandas as pd

mlflow.set_registry_uri("databricks-uc")

model_uri = "models:/dev.bronze.travel_itinerary/1"

travel_model_v1 = mlflow.pyfunc.load_model(model_uri)

print("Model loaded successfully!")
print(model_uri)

In [0]:
test_input = pd.DataFrame([
    {
        "city": "Jaipur",
        "month": "August",
        "budget": 1000.0,
        "days": 3,
        "num_people": 2
    }
])

result = travel_model_v1.predict(test_input)

print(result[0])

In [0]:
from mlflow import MlflowClient

client = MlflowClient()

versions = client.search_model_versions(
    "name = 'dev.bronze.travel_itinerary'"
)

for version in versions:
    print(
        "Version:",
        version.version,
        "| Run ID:",
        version.run_id
    )